In [ ]:
# 09 – Final Playing XI Assembly

Purpose:
- Assemble a balanced Playing XI
- Enforce role constraints
- Remove duplicate players

Input:
- Ranked role-based candidates (Notebook 08)

Output:
- data/processed/final_playing_xi.csv

In [6]:
# =========================================================
# 09 – Final Playing XI Assembly (FULL CLEAN SINGLE CELL)
# =========================================================

import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from config import *
import pandas as pd


# ---------------------------------------------------------
# Load Data
# ---------------------------------------------------------

top_openers = pd.read_csv(OUTPUTS_DIR / "tables/top_openers.csv")
top_middle = pd.read_csv(OUTPUTS_DIR / "tables/top_middle.csv")
top_finishers = pd.read_csv(OUTPUTS_DIR / "tables/top_finishers.csv")
top_all_rounders = pd.read_csv(OUTPUTS_DIR / "tables/top_all_rounders.csv")

first_bowlers = pd.read_csv(PROCESSED_DIR / "first_bowlers.csv")
middle_bowlers = pd.read_csv(PROCESSED_DIR / "middle_bowlers.csv")
death_bowlers = pd.read_csv(PROCESSED_DIR / "death_bowlers.csv")


# ---------------------------------------------------------
# Initial Role Selection
# ---------------------------------------------------------

final_openers = top_openers.head(2).copy()
final_middle = top_middle.head(2).copy()
final_finishers = top_finishers.head(2).copy()
final_all_rounders = top_all_rounders.head(2).copy()

final_pp_bowler = first_bowlers.head(1).copy()
final_middle_bowler = middle_bowlers.head(1).copy()
final_death_bowler = death_bowlers.head(1).copy()


# ---------------------------------------------------------
# Assign Roles
# ---------------------------------------------------------

final_openers["player"] = final_openers["batsman"]
final_openers["role"] = "Opener"

final_middle["player"] = final_middle["batsman"]
final_middle["role"] = "Middle Order"

final_finishers["player"] = final_finishers["batsman"]
final_finishers["role"] = "Finisher"

final_all_rounders["role"] = "All-Rounder"

final_pp_bowler["player"] = final_pp_bowler["bowler"]
final_pp_bowler["role"] = "Powerplay Bowler"

final_middle_bowler["player"] = final_middle_bowler["bowler"]
final_middle_bowler["role"] = "Middle Overs Bowler"

final_death_bowler["player"] = final_death_bowler["bowler"]
final_death_bowler["role"] = "Death Bowler"


# ---------------------------------------------------------
# Combine XI
# ---------------------------------------------------------

playing_xi = pd.concat([
    final_openers[["player","role"]],
    final_middle[["player","role"]],
    final_finishers[["player","role"]],
    final_all_rounders[["player","role"]],
    final_pp_bowler[["player","role"]],
    final_middle_bowler[["player","role"]],
    final_death_bowler[["player","role"]],
], ignore_index=True)

playing_xi = playing_xi.drop_duplicates("player").reset_index(drop=True)


# ---------------------------------------------------------
# Role Targets
# ---------------------------------------------------------

ROLE_TARGETS = {
    "Opener":2,
    "Middle Order":2,
    "Finisher":2,
    "All-Rounder":2,
    "Powerplay Bowler":1,
    "Middle Overs Bowler":1,
    "Death Bowler":1
}

pools = {
    "Opener": (top_openers,"batsman"),
    "Middle Order": (top_middle,"batsman"),
    "Finisher": (top_finishers,"batsman"),
    "All-Rounder": (top_all_rounders,"player"),
    "Powerplay Bowler": (first_bowlers,"bowler"),
    "Middle Overs Bowler": (middle_bowlers,"bowler"),
    "Death Bowler": (death_bowlers,"bowler")
}

selected = set(playing_xi["player"])

for role,target in ROLE_TARGETS.items():

    current = playing_xi[playing_xi["role"]==role].shape[0]
    missing = target-current

    if missing>0:

        pool,col = pools[role]

        candidates = pool[~pool[col].isin(selected)]

        for i in range(min(missing,len(candidates))):

            p = candidates.iloc[i][col]

            playing_xi = pd.concat([
                playing_xi,
                pd.DataFrame([{"player":p,"role":role}])
            ],ignore_index=True)

            selected.add(p)


# ---------------------------------------------------------
# Enforce Max 4 Foreign Players
# ---------------------------------------------------------

foreign_players = {
    "JC Buttler","AB de Villiers","KA Pollard","SP Narine",
    "DJ Bravo","AD Russell","JM Bairstow","DA Warner",
    "RG Maxwell","F du Plessis","N Pooran","Rashid Khan","SL Malinga"
}

playing_xi["is_foreigner"] = playing_xi["player"].isin(foreign_players)

MAX_FOREIGNERS = 4

if playing_xi["is_foreigner"].sum() > MAX_FOREIGNERS:

    removal_priority = {
        "Death Bowler":1,
        "Middle Overs Bowler":2,
        "Powerplay Bowler":3,
        "All-Rounder":4,
        "Finisher":5,
        "Middle Order":6,
        "Opener":7
    }

    foreigners = playing_xi[playing_xi["is_foreigner"]].copy()
    foreigners["priority"] = foreigners["role"].map(removal_priority)

    remove_player = foreigners.sort_values("priority").iloc[0]
    role = remove_player["role"]

    playing_xi = playing_xi[
        playing_xi["player"] != remove_player["player"]
    ]

    pool,col = pools[role]

    replacement = pool[
        (~pool[col].isin(playing_xi["player"])) &
        (~pool[col].isin(foreign_players))
    ].iloc[0][col]

    playing_xi = pd.concat([
        playing_xi,
        pd.DataFrame([{"player":replacement,"role":role}])
    ],ignore_index=True)


# ---------------------------------------------------------
# Final Ordering
# ---------------------------------------------------------

role_order = [
"Opener",
"Middle Order",
"Finisher",
"All-Rounder",
"Powerplay Bowler",
"Middle Overs Bowler",
"Death Bowler"
]

playing_xi["rank"] = playing_xi["role"].apply(lambda x:role_order.index(x))

playing_xi = playing_xi.sort_values("rank").drop(columns=["rank","is_foreigner"])

playing_xi = playing_xi.reset_index(drop=True)


# ---------------------------------------------------------
# Save Output
# ---------------------------------------------------------

playing_xi.to_csv(PROCESSED_DIR/"final_playing_xi.csv",index=False)

print("Final Playing XI\n")
print(playing_xi)

Final Playing XI

            player                 role
0       JC Buttler               Opener
1      YBK Jaiswal               Opener
2          V Kohli         Middle Order
3         SK Raina         Middle Order
4   AB de Villiers             Finisher
5       KA Pollard             Finisher
6        RA Jadeja          All-Rounder
7        SP Narine          All-Rounder
8          B Kumar     Powerplay Bowler
9         R Ashwin  Middle Overs Bowler
10       JJ Bumrah         Death Bowler
